[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LizbethMG-Teaching/pose2behav-book/blob/main/notebooks/prepare-multi-animal-data.ipynb)

In [ ]:

# PREFILLED, NO NEED TO CHANGE, JUST RUN THIS CELL
# Install and import the required libraries:
!pip -q install gdown tables

import os
from pathlib import Path
import gdown, pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import groupby
import re
import numpy as np
from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle


# --------------------------------------------------------------

# Detect if running in Google Colab
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    DEST = Path("/content/cleaned_pose_downloaded.h5")
else:
    DEST = Path("cleaned_pose_downloaded.h5")  # save in current folder locally
print("Saving to:", DEST)

# Select here the experiment you want to download, comment the others:
# mice-5_5min (a bit noisy, only 3 mice tracked well)
#FILE_ID = "15Eiib-vpdunmxzYiP3RfKL0_iur-dCSJ"
# mice-3-4min (cleaner, but less data)
FILE_ID = "16s9ZduNwU0oOOHyqcU8ESm8FZA3TEvBi"

URL = f"https://drive.google.com/uc?id={FILE_ID}"

print("Downloading from Drive...")
_ = gdown.download(URL, str(DEST), quiet=False)

# Basic checks
assert DEST.exists() and DEST.stat().st_size > 0, "❌ Download failed or empty file."
print(f"✅ Downloaded to {DEST} ({DEST.stat().st_size/1_000_000:.2f} MB)")

# Name of cleaned output (3 mice)
CLEAN_H5_NAME = "cleaned_pose_3mice_5min.h5"

# Load the H5 file into a pandas DataFrame
df_multi = pd.read_hdf(DEST, key="df_with_missing")

print("✅ Data loaded successfully!")
print("Shape:", df_multi.shape)
print("Columns:", list(df_multi.columns)[:8], "...")

print("Column level names:", df_multi.columns.names)


# =========================
# 3. Split by animal and flatten columns
# =========================

def extract_single_animal(df_multi: pd.DataFrame, animal_name: str) -> pd.DataFrame:
    """
    From a DLC multi-animal output with column levels
    (scorer, individuals, bodyparts, coords),
    extract one animal and produce flat columns like:
    'nose_x', 'nose_y', 'nose_likelihood', etc.
    """
    cols = df_multi.columns
    col_levels = list(cols.names)

    if "individuals" not in col_levels:
        raise ValueError("Expected a MultiIndex with level 'individuals', got levels: "
                         f"{col_levels}")

    # Select this animal on the 'individuals' level
    sub = df_multi.xs(animal_name, axis=1, level="individuals")

    # Drop scorer if present
    if "scorer" in sub.columns.names:
        sub = sub.droplevel("scorer", axis=1)

    # After dropping scorer, we expect levels like ('bodyparts', 'coords')
    # Build flat names: bodypart_coord
    if len(sub.columns.names) == 2:
        flat_names = []
        for bp, coord in sub.columns.to_list():
            flat_names.append(f"{bp}_{coord}")
        sub.columns = flat_names
    else:
        # Fallback: just join all levels with underscores
        sub.columns = ["_".join(map(str, c)) for c in sub.columns.to_list()]

    return sub


def get_animals(df_multi: pd.DataFrame):
    """Return the list of animal track IDs from the 'individuals' column level."""
    if not isinstance(df_multi.columns, pd.MultiIndex):
        raise ValueError("Expected a MultiIndex for columns")

    if "individuals" not in df_multi.columns.names:
        raise ValueError(f"'individuals' level not found in column names {df_multi.columns.names}")

    indiv_level = df_multi.columns.names.index("individuals")
    animals = df_multi.columns.get_level_values(indiv_level).unique().tolist()
    return animals


animals = get_animals(df_multi)
print("Animals found:", animals)

# Build flat single-animal DataFrames
single_animal_dfs = {a: extract_single_animal(df_multi, a) for a in animals}
for a, dfa in single_animal_dfs.items():
    print(f"{a}: shape {dfa.shape}")


# =========================
# 4. Softer quality summary per animal (to choose best 3)
# =========================

def compute_animal_quality(df_animal: pd.DataFrame, lik_thresh: float = 0.7) -> float:
    """
    Softer quality metric per animal.

    Fraction of frames where a small set of keypoints
    (mouse_center, tail_base, head_midpoint if present)
    have:
      - likelihood >= lik_thresh
      - x and y not equal to -1

    If none of these keypoints exist, fall back to the strict
    "all bodyparts good" rule, but still with lik_thresh.
    """
    key_parts = ["mouse_center", "tail_base", "head_midpoint"]

    kp_lik_cols = []
    kp_coord_cols = []

    for bp in key_parts:
        xcol = f"{bp}_x"
        ycol = f"{bp}_y"
        lcol = f"{bp}_likelihood"
        if xcol in df_animal.columns:
            kp_coord_cols.append(xcol)
        if ycol in df_animal.columns:
            kp_coord_cols.append(ycol)
        if lcol in df_animal.columns:
            kp_lik_cols.append(lcol)

    if kp_lik_cols and kp_coord_cols:
        df_lik = df_animal[kp_lik_cols]
        df_coord = df_animal[kp_coord_cols]
    else:
        # Fallback: use all bodyparts, but with lower threshold
        lik_cols = [c for c in df_animal.columns if c.endswith("_likelihood")]
        coord_cols = [c for c in df_animal.columns if c.endswith("_x") or c.endswith("_y")]
        if not lik_cols or not coord_cols:
            return 0.0
        df_lik = df_animal[lik_cols]
        df_coord = df_animal[coord_cols]

    lik_ok = (df_lik >= lik_thresh).all(axis=1)
    coord_ok = (df_coord != -1).all(axis=1)

    valid_frames = lik_ok & coord_ok
    return valid_frames.mean() * 100.0  # percentage


summary = []
for a, dfa in single_animal_dfs.items():
    q = compute_animal_quality(dfa, lik_thresh=0.7)
    summary.append({"animal": a, "quality_percent": q})

summary_df = (
    pd.DataFrame(summary)
    .set_index("animal")
    .sort_values("quality_percent", ascending=False)
)

print("\nQuality summary (softer metric, % of good frames):")
print(summary_df)

# Number of best animals (we now take only the best 3)
N_BEST = 3
best_animals = summary_df.index[:N_BEST].tolist()
print(f"\nUsing best {N_BEST} animals:", best_animals)


# =========================
# 5. Cleaning utilities
# =========================

def percent_nans(df: pd.DataFrame) -> pd.Series:
    """Percent of NaNs per column."""
    return df.isna().mean() * 100.0


def compute_bodypart_speeds(df_animal: pd.DataFrame) -> dict:
    """
    Compute per-bodypart speed (pixels per frame) time series.
    Returns {bodypart_name: pd.Series of speeds}.
    Assumes columns like 'nose_x', 'nose_y', ...
    """
    speeds = {}
    bp_names = sorted({c[:-2] for c in df_animal.columns if c.endswith("_x")})
    for bp in bp_names:
        x = df_animal[f"{bp}_x"].astype(float)
        y = df_animal[f"{bp}_y"].astype(float)
        dx = x.diff()
        dy = y.diff()
        s = np.sqrt(dx * dx + dy * dy)
        speeds[bp] = s
    return speeds


def compute_mad_thresholds(df_animal: pd.DataFrame, k: float = 3.5) -> dict:
    """
    For each bodypart, compute a jump threshold based on median + k * MAD of speed.
    Returns {bodypart_name: threshold_pixels_per_frame}
    """
    speeds = compute_bodypart_speeds(df_animal)
    thresholds = {}
    for bp, s in speeds.items():
        s_valid = s.replace([np.inf, -np.inf], np.nan).dropna()
        if len(s_valid) < 10:
            thresholds[bp] = np.inf
            continue
        med = s_valid.median()
        mad = (s_valid - med).abs().median()
        thr = med + k * mad
        thresholds[bp] = float(thr)
    return thresholds


def build_mask_neg1_and_jumps(
    df_animal: pd.DataFrame,
    thresholds: dict,
    max_speed_px_per_frame: float = 80.0,
) -> pd.DataFrame:
    """
    Build a boolean mask for bad samples:
    - coordinates equal to -1
    - speed larger than per-bodypart threshold or larger than max_speed_px_per_frame
    Returns a DataFrame mask with same shape as df_animal (True = bad to mask).
    """
    mask = pd.DataFrame(False, index=df_animal.index, columns=df_animal.columns)

    # 1) mask -1 coordinates
    coord_cols = [c for c in df_animal.columns if c.endswith("_x") or c.endswith("_y")]
    neg1_mask = df_animal[coord_cols] == -1
    mask.loc[:, coord_cols] |= neg1_mask

    # 2) mask jumps based on speed
    speeds = compute_bodypart_speeds(df_animal)
    for bp, s in speeds.items():
        thr = thresholds.get(bp, np.inf)
        jump_thr = min(thr, max_speed_px_per_frame)
        jump_idx = s > jump_thr
        if not jump_idx.any():
            continue
        bad_frames = jump_idx[jump_idx].index
        cols = [f"{bp}_x", f"{bp}_y", f"{bp}_likelihood"]
        cols = [c for c in cols if c in df_animal.columns]
        mask.loc[bad_frames, cols] = True

    return mask


def apply_mask(df_animal: pd.DataFrame, mask: pd.DataFrame) -> pd.DataFrame:
    """Set masked entries to NaN and return a new DataFrame."""
    out = df_animal.copy()
    out[mask] = np.nan
    return out


def interpolate_short_gaps(df_animal: pd.DataFrame, max_gap: int = 10) -> pd.DataFrame:
    """
    Interpolate NaN gaps up to length max_gap.
    Uses linear interpolation frame-wise.
    """
    out = df_animal.copy()
    for col in out.columns:
        s = out[col]
        if not np.issubdtype(s.dtype, np.number):
            continue
        out[col] = s.interpolate(
            method="linear",
            limit=max_gap,
            limit_direction="both"
        )
    return out


def smooth_positions(df_animal: pd.DataFrame, window: int = 5) -> pd.DataFrame:
    """
    Apply a rolling mean to position columns (x,y).
    Does not smooth likelihood columns.
    """
    out = df_animal.copy()
    for col in out.columns:
        if col.endswith("_x") or col.endswith("_y"):
            out[col] = (
                out[col]
                .rolling(window=window, center=True, min_periods=1)
                .mean()
            )
    return out


# =========================
# 6. Cleaning and saving the best N animals (N_BEST = 3)
# =========================

MAD_K = 4
MAX_SPEED_PX_PER_FRAME = 100.0
MAX_SHORT_GAP = 20
SMOOTH_WINDOW = 5
APPLY_SMOOTHING = True

cleaned_list = []

for animal in best_animals:
    print(f"\n=== Cleaning {animal} ===")
    df_animal = single_animal_dfs[animal].copy()

    # Convert -1 likelihoods to 0 (optional, but helps quality metrics)
    lik_cols = [c for c in df_animal.columns if c.endswith("_likelihood")]
    for col in lik_cols:
        df_animal[col] = df_animal[col].replace(-1, 0.0)

    # 1) thresholds
    thresholds = compute_mad_thresholds(df_animal, k=MAD_K)
    print("Speed thresholds (px/frame):")
    for bp, thr in sorted(thresholds.items()):
        print(f"  {animal} / {bp:>12s}: {thr:7.2f}")

    # 2) build mask and apply
    mask = build_mask_neg1_and_jumps(
        df_animal,
        thresholds=thresholds,
        max_speed_px_per_frame=MAX_SPEED_PX_PER_FRAME,
    )
    df_stage = apply_mask(df_animal, mask)

    print("Percent NaNs after masking:")
    print(percent_nans(df_stage))

    # 3) interpolate short gaps
    df_stage = interpolate_short_gaps(df_stage, max_gap=MAX_SHORT_GAP)

    # 4) optional smoothing
    if APPLY_SMOOTHING:
        df_stage = smooth_positions(df_stage, window=SMOOTH_WINDOW)

    print("Percent NaNs after interpolation:")
    print(percent_nans(df_stage))

    # Prefix columns with animal id so they stay unique after concat
    df_stage = df_stage.add_prefix(f"{animal}_")
    cleaned_list.append(df_stage)

# Concatenate all cleaned animals side by side
df_clean_all = pd.concat(cleaned_list, axis=1)
print("\nFinal cleaned shape (rows, columns):", df_clean_all.shape)

# Save to HDF5
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    out_path = Path("/content") / CLEAN_H5_NAME
else:
    out_path = Path(CLEAN_H5_NAME)

df_clean_all.to_hdf(out_path, key="df_with_missing", mode="w")
print(f"Saved cleaned multi-animal file to: {out_path}")